# TECO — all plots from the completed run
**CPU → Runtime → Run all**, then allow Drive mounting. Reads your existing `teco_v1_word_erp_compact` compact results. It does not run extraction, change labels, match fixations, reload source EEG or save trial arrays.

Shows **overall scalp summaries and all six scalp regions**, followed by **every subject's combined overall + regional figure**. ZuCo keeps SR/NR/TSR separate and includes the original 12-channel overview. TeCo includes pooled neutral, strict neutral-context and original source-rated neutral comparisons, with each comparison's matched readers recorded.

**A full-head signed mean should be almost zero after average reference.** This notebook shows it honestly, and also plots **global field power (GFP)** separately: channel-wise population standard deviation of each reader's class-average waveform, then averaged across readers. GFP is nonnegative, not signed ERP voltage or single-trial GFP. Unequal trial counts and residual noise can influence it. All figures remain descriptive.

New plot files and small CSV reports stay in the existing result folder under `analysis/all_plots_v1/`. Expect a few to a few tens of MB of additional Drive storage, depending on image compression. Running again replaces this gallery's generated files; source averages and extraction checkpoints are read only. No GitHub token is needed. Code stays on GitHub and in temporary Colab storage.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json
from IPython.display import display, Markdown, Image

drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/Thesis/Results/word_sentiment_eeg/teco_v1_word_erp_compact')
REPO = Path('/content/word-sentiment-eeg-teco-plots')
if not (RUN/'manifest.json').is_file():
    raise FileNotFoundError(f'Saved results not found: {RUN}. Check the mounted Google account. No extraction will start.')
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary plotting checkout has edits. Preserve those before updating.')
    git('-C',REPO,'pull','--ff-only','origin','main')
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','pandas','matplotlib'],check=True)
print('Code version:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/plot_teco.py'),'--run-dir',str(RUN)]
run_with_progress(command,RUN/'all_plots_progress.json',activity='TECO: overall, GFP, scalp-region and reader plots')
PLOTS=RUN/'analysis/all_plots_v1'
GALLERY=json.loads((PLOTS/'gallery.json').read_text())


In [ ]:
display(Markdown((PLOTS/'REPORT.md').read_text()))
print('Saved gallery:', PLOTS)
for item in GALLERY:
    if item['kind']=='group':
        display(Markdown('### '+item['title']))
        display(Image(filename=str(PLOTS/item['path'])))
display(Markdown('### Channel assignments'))
display(Image(filename=str(PLOTS/'sensor_regions.png')))


In [ ]:
# Every subject: overall and six scalp areas together in the same figure.
for item in GALLERY:
    if item['kind']=='subject':
        display(Markdown('### '+item['title']))
        display(Image(filename=str(PLOTS/item['path'])))
print('All image paths:', PLOTS/'gallery.json')
print('Curve values:', PLOTS/'group_curves.csv.gz')
print('Readers in each comparison:', PLOTS/'comparison_rosters.json')
